# 01 — LSTM Sentiment Analysis on IMDB Movie Reviews
### Deep Learning Week 3: Recurrent Neural Networks & Sequential Modeling

---

## 1. Conceptual Introduction: The Need for Recurrent Networks

Traditional feedforward neural networks assume all inputs and outputs are independent of each other. However, human language is inherently **sequential**: the meaning of a word depends heavily on the words that precede and follow it.

### Vanilla Recurrent Neural Networks (RNNs)
A standard RNN maintains an internal hidden state $h_t$ that carries information across time steps $t$:
$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$

While conceptually elegant, vanilla RNNs suffer from **vanishing and exploding gradients** during backpropagation through time (BPTT). When computing gradients over long sequences, repeated matrix multiplication with weight matrices $W_{hh}$ causes gradients to either decay exponentially to zero (preventing the model from learning long-term dependencies) or grow uncontrollably to infinity.

### Long Short-Term Memory (LSTM) Networks
Introduced by Hochreiter & Schmidhuber (1997), **LSTMs** solve the vanishing gradient problem through a dedicated **cell state** $C_t$ (the "constant error carousel") regulated by three multiplicative gating mechanisms:

1. **Forget Gate ($f_t$):** Determines what proportion of past cell state information should be discarded:
   $$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
2. **Input Gate ($i_t$ & $\tilde{C}_t$):** Decides which new information to store in the cell state:
   $$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
   $$\tilde{C}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$$
3. **Cell State Update ($C_t$):** Combines retained past memory with newly gated candidate memory:
   $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$
4. **Output Gate ($o_t$ & $h_t$):** Controls what portion of the cell state is emitted as the new hidden state:
   $$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
   $$h_t = o_t \odot \tanh(C_t)$$

In this notebook, we apply an LSTM to binary sentiment classification on the 50,000-review **IMDB dataset**.


In [ ]:
# 2. Imports and Environment Verification
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Set reproducible random seed
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")
print(f"GPU Available:      {len(tf.config.list_physical_devices('GPU')) > 0}")


## 3. Dataset Loading & Inspection: IMDB Movie Reviews

The IMDB dataset consists of 50,000 highly polarized movie reviews labeled as **0 (Negative)** or **1 (Positive)**.
- **25,000 Training Reviews**
- **25,000 Held-Out Testing Reviews**

In Keras, reviews are pre-tokenized into integers where each number represents the frequency rank of that word in the dataset.


In [ ]:
# 3.1 Load IMDB Dataset with Top 10,000 Words
VOCAB_SIZE = 10000
MAX_LEN = 200  # Cut off reviews after 200 words or pad if shorter

print(f"Loading IMDB dataset (top {VOCAB_SIZE:,} words)...")
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

print(f"Train samples: {len(x_train_raw):,}")
print(f"Test samples:  {len(x_test_raw):,}")

# Inspect Class Distribution
pos_train = np.sum(y_train_raw == 1)
neg_train = np.sum(y_train_raw == 0)
print(f"Train Split -> Positive: {pos_train:,} ({pos_train/len(y_train_raw)*100:.1f}%), Negative: {neg_train:,} ({neg_train/len(y_train_raw)*100:.1f}%)")


### 3.2 Inspecting Token IDs and Review Lengths

In the Keras IMDB dataset:
- Index `0` is reserved for **Padding** (`<PAD>`)
- Index `1` represents the **Start of Sequence** token (`<START>`)
- Index `2` represents **Out of Vocabulary / Unknown Words** (`<UNK>`)
- Index `3` is reserved for an unused token, so actual word ranks begin at index `4`.


In [ ]:
# Retrieve the word index dictionary to decode reviews back to text
word_index = keras.datasets.imdb.get_word_index()
# Invert dictionary and offset indices by 3
inverted_word_index = {i + 3: word for word, i in word_index.items()}
inverted_word_index[0] = "<PAD>"
inverted_word_index[1] = "<START>"
inverted_word_index[2] = "<UNK>"
inverted_word_index[3] = "<UNUSED>"

def decode_review(token_list):
    return " ".join([inverted_word_index.get(i, "?") for i in token_list])

print("Sample Tokenized Review (first 20 token IDs):", x_train_raw[0][:20])
print(f"Label: {'Positive (1)' if y_train_raw[0] == 1 else 'Negative (0)'}")
print("\nDecoded Review (Excerpt):")
print(decode_review(x_train_raw[0][:50]) + "...")

# Review Length Distribution
review_lengths = [len(r) for r in x_train_raw]
print(f"\nReview Lengths: Min={min(review_lengths)}, Max={max(review_lengths)}, Mean={np.mean(review_lengths):.1f}, Median={np.median(review_lengths)}")


## 4. Sequence Preprocessing: Padding & Truncation

Neural networks operate on rectangular tensor batches. Because natural language reviews vary from 10 to over 2,000 words, we standardize input length to `MAX_LEN = 200`:
- Shorter reviews are **post-padded** with zeros.
- Longer reviews are **post-truncated** to the first 200 tokens.


In [ ]:
# 4.1 Pad and Truncate Sequences
x_train_padded = pad_sequences(x_train_raw, maxlen=MAX_LEN, padding="post", truncating="post")
x_test_padded = pad_sequences(x_test_raw, maxlen=MAX_LEN, padding="post", truncating="post")

# Derive a 20% validation split from training data (keeping test set completely untouched)
VAL_SPLIT = 5000
x_val = x_train_padded[:VAL_SPLIT]
y_val = y_train_raw[:VAL_SPLIT]
x_train = x_train_padded[VAL_SPLIT:]
y_train = y_train_raw[VAL_SPLIT:]

print(f"Training shape:   {x_train.shape}, labels: {y_train.shape}")
print(f"Validation shape: {x_val.shape}, labels: {y_val.shape}")
print(f"Held-out test:    {x_test_padded.shape}, labels: {y_test_raw.shape}")


## 5. Model Architecture: Embedding + LSTM Classifier

### Architecture Layers:
1. **Embedding Layer (`input_dim=10000`, `output_dim=64`, `input_length=200`):**
   Converts discrete integer token indices into dense continuous vector representations. Words with similar semantic meanings and syntactic roles are mapped close together in this 64-dimensional learned latent space.
2. **SpatialDropout1D (`0.2`):**
   Drops entire 1D feature maps across the sequence, preventing co-adaptation of word features.
3. **LSTM Layer (`units=64`, `dropout=0.2`):**
   Processes the 200-step sequence sequentially, maintaining cell state $C_t$ and emitting the final summary hidden vector $h_{200} \in \mathbb{R}^{64}$.
4. **Dense Output Layer (`units=1`, `activation='sigmoid'`):**
   Computes the probability that the review expresses positive sentiment: $P(y=1 | x) = \sigma(W \cdot h_{200} + b)$.


In [ ]:
# 5.1 Build the LSTM Model
EMBEDDING_DIM = 64
LSTM_UNITS = 64

model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,), dtype=tf.int32, name="token_input"),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, name="word_embedding"),
    layers.SpatialDropout1D(0.2, name="spatial_dropout"),
    layers.LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.0, name="lstm_classifier"),
    layers.Dense(32, activation="relu", name="dense_intermediate"),
    layers.Dropout(0.3, name="dense_dropout"),
    layers.Dense(1, activation="sigmoid", name="sentiment_output"),
], name="IMDB_LSTM_Classifier")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 6. Training with Early Stopping & Validation Tracking

We train for up to 8 epochs with early stopping monitoring validation loss to prevent overfitting.


In [ ]:
# 6.1 Train the Model
EPOCHS = 6
BATCH_SIZE = 128

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-4,
        verbose=1,
    )
]

history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)


## 7. Training Diagnostics & Learning Curves


In [ ]:
# 7.1 Plot Training and Validation History
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history["loss"], label="Train Loss", marker="o", color="#1f77b4")
ax1.plot(history.history["val_loss"], label="Val Loss", marker="s", color="#ff7f0e")
ax1.set_title("Binary Cross-Entropy Loss Across Epochs", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.legend()

# Accuracy
ax2.plot(history.history["accuracy"], label="Train Accuracy", marker="o", color="#2ca02c")
ax2.plot(history.history["val_accuracy"], label="Val Accuracy", marker="s", color="#d62728")
ax2.set_title("Classification Accuracy Across Epochs", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.grid(True, linestyle="--", alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()


## 8. Final Evaluation on Held-Out Test Set


In [ ]:
# 8.1 Evaluate on Official 25,000 Test Reviews
test_loss, test_accuracy = model.evaluate(x_test_padded, y_test_raw, batch_size=256, verbose=1)
print(f"\n================ HELD-OUT TEST RESULTS ================")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"========================================================")


## 9. Sample Predictions & Error Analysis

We inspect positive, negative, and borderline misclassified reviews to analyze model behavior.


In [ ]:
# 9.1 Run Inference on Sample Test Reviews
sample_indices = [0, 1, 2, 3, 4, 10, 15, 20]
test_preds = model.predict(x_test_padded[sample_indices], verbose=0).flatten()

print("=" * 80)
print(f"{'Idx':<4} | {'True Label':<10} | {'Pred Prob':<10} | {'Predicted':<10} | {'Match?':<6}")
print("=" * 80)
for idx, p_prob in zip(sample_indices, test_preds):
    true_lbl = "Positive" if y_test_raw[idx] == 1 else "Negative"
    pred_lbl = "Positive" if p_prob >= 0.5 else "Negative"
    match = "MATCH" if true_lbl == pred_lbl else "MISMATCH"
    print(f"{idx:<4} | {true_lbl:<10} | {p_prob:9.4f} | {pred_lbl:<10} | {match:<8}")

# 9.2 Inspect Misclassified Review
all_test_preds = model.predict(x_test_padded[:500], verbose=0).flatten()
misclassified_indices = np.where((all_test_preds >= 0.5) != y_test_raw[:500])[0]

if len(misclassified_indices) > 0:
    err_idx = misclassified_indices[0]
    err_prob = all_test_preds[err_idx]
    err_true = "Positive" if y_test_raw[err_idx] == 1 else "Negative"
    err_pred = "Positive" if err_prob >= 0.5 else "Negative"
    print(f"\n--- Detailed Misclassification Case (Index {err_idx}) ---")
    print(f"True Label: {err_true} | Predicted: {err_pred} (Confidence: {err_prob:.4f})")
    print("Decoded Text:")
    print(decode_review(x_test_raw[err_idx][:100]) + "...")


## 10. Discussion & Limitations

### 1. Strengths of the LSTM Classifier:
- Captures sequential context and word order far better than bag-of-words or naive Bayes models.
- Word embeddings enable the model to generalize between synonyms without manual feature engineering.

### 2. Limitations & Failure Modes:
- **Sequence Truncation:** Cutting off reviews at 200 words discards critical concluding sentiment in longer reviews.
- **Vocabulary Truncation:** Words outside the top 10,000 frequency ranks become `<UNK>`, which may remove specific actor names, technical terms, or rare adjectives.
- **Sarcasm and Nuance:** Binary sentiment labels struggle with sarcastic reviews (e.g., *"A masterpiece of sheer boredom"*), which contain positive words used with negative intent.
- **Bi-directionality:** Standard forward LSTMs only see preceding words. A Bidirectional LSTM (BiLSTM) would improve performance by conditioning each word on both prior and subsequent context.
